# Notebook 49 — Adaptive Serving Infrastructure

**Engineering statement:** Adaptive serving infrastructure turns operating-regime decisions into deployable routing, scaling, fallback, and telemetry loops.

Notebook 43 mapped operating regimes to resource-allocation policies. Notebook 49 moves one level closer to deployment: routing, replica pools, autoscaling, saturation fallback, and feedback.

The core specification is:

\[
\text{operating regime} \rightarrow \text{controller policy} \rightarrow \text{routing state} \rightarrow \text{replica allocation} \rightarrow \text{telemetry feedback}.
\]

In [ ]:
# Notebook 49 setup
from pathlib import Path
import json
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from matplotlib.colors import ListedColormap, BoundaryNorm
from IPython.display import display, FileLink

ROOT = Path.cwd()
FIG_DIR = ROOT / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = ROOT / "results" / "notebook49_results.zip"

plt.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 200,
    "font.size": 12,
    "axes.titlesize": 24,
    "axes.labelsize": 14,
    "legend.fontsize": 11,
})

def savefig(name):
    path = FIG_DIR / name
    plt.savefig(path, bbox_inches="tight")
    plt.show()
    return path

def rounded_box(ax, xy, w, h, text, fontsize=12, lw=1.6):
    box = FancyBboxPatch(
        xy, w, h,
        boxstyle="round,pad=0.02,rounding_size=0.04",
        facecolor="white",
        edgecolor="black",
        linewidth=lw
    )
    ax.add_patch(box)
    ax.text(xy[0] + w/2, xy[1] + h/2, text,
            ha="center", va="center", fontsize=fontsize)
    return box

def arrow(ax, start, end, lw=1.6, rad=0.0):
    arr = FancyArrowPatch(
        start, end,
        arrowstyle="->",
        mutation_scale=12,
        linewidth=lw,
        color="black",
        connectionstyle=f"arc3,rad={rad}"
    )
    ax.add_patch(arr)
    return arr

def hide(ax):
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

## 1. Repository transition

Notebook 49 continues the second systems arc: after resource allocation, the controller becomes infrastructure.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 7))
hide(ax)
ax.set_title("Adaptive infrastructure continues the second systems arc", pad=24)

top = [
    ("00\nContext", 0.04), ("07\nVerification\nResource", 0.18),
    ("13\nConfidence\nScheduling", 0.32), ("17\nSemi-AR\nDrafting", 0.46),
    ("23\nThroughput\nObjective", 0.60), ("29\nHardware\nConstraints", 0.74),
    ("37\nOperating\nRegimes", 0.88)
]
w, h = 0.12, 0.12
for label, x in top:
    rounded_box(ax, (x-w/2, 0.70), w, h, label, fontsize=9)
for (_, x1), (_, x2) in zip(top[:-1], top[1:]):
    arrow(ax, (x1+w/2, 0.76), (x2-w/2, 0.76))

ax.text(0.50, 0.60, "first arc: mechanism → operating regime", ha="center", va="center", fontsize=13)
ax.plot([0.08, 0.92], [0.54, 0.54], color="black", lw=1.2)

bottom = [
    ("43\nResource\nAllocation", 0.38),
    ("49\nAdaptive\nInfrastructure", 0.52),
    ("53\nDistributed\nScheduling", 0.66),
    ("59\nCluster\nOptimization", 0.80),
]
for label, x in bottom:
    rounded_box(ax, (x-w/2, 0.32), w, h, label, fontsize=9)
for (_, x1), (_, x2) in zip(bottom[:-1], bottom[1:]):
    arrow(ax, (x1+w/2, 0.38), (x2-w/2, 0.38))

ax.text(0.59, 0.20, "second arc: controller → infrastructure → distributed scheduling", ha="center", va="center", fontsize=13)

savefig("49_repository_transition.png")

## 2. Infrastructure map

The deployment object is not just a scheduler. It is a serving infrastructure that routes requests, allocates replicas, observes telemetry, and updates policy.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 7))
hide(ax)
ax.set_title("Adaptive serving infrastructure map", pad=24)

# top serving path
nodes = [
    ("Requests", 0.10, 0.72),
    ("Ingress\nqueue", 0.26, 0.72),
    ("Runtime\nrouter", 0.42, 0.72),
    ("Replica\npool", 0.62, 0.72),
    ("Target\nverify", 0.82, 0.72),
]
w, h = 0.13, 0.11
for label, x, y in nodes:
    rounded_box(ax, (x-w/2, y-h/2), w, h, label, fontsize=11)
for (_, x1, y1), (_, x2, y2) in zip(nodes[:-1], nodes[1:]):
    arrow(ax, (x1+w/2, y1), (x2-w/2, y2))

# lower control path
rounded_box(ax, (0.18, 0.38), 0.16, 0.10, "Telemetry\nstate", fontsize=10)
rounded_box(ax, (0.40, 0.36), 0.24, 0.13, "Operating-regime\nclassifier", fontsize=10)
rounded_box(ax, (0.68, 0.38), 0.16, 0.10, "Autoscaler", fontsize=10)
rounded_box(ax, (0.48, 0.16), 0.16, 0.10, "Policy\ncache", fontsize=10)

arrow(ax, (0.42, 0.66), (0.52, 0.49))
arrow(ax, (0.34, 0.43), (0.40, 0.43))
arrow(ax, (0.64, 0.43), (0.68, 0.43))
arrow(ax, (0.76, 0.48), (0.62, 0.66), rad=0.18)
arrow(ax, (0.52, 0.36), (0.56, 0.26))
arrow(ax, (0.56, 0.26), (0.62, 0.66), rad=-0.16)

ax.text(0.50, 0.08,
        "Infrastructure turns controller policy into routing, scaling, and verification placement.",
        ha="center", va="center", fontsize=13)

savefig("49_infrastructure_map.png")

## 3. Runtime routing controller

At runtime, request metadata and live system state jointly select the route.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))
hide(ax)
ax.set_title("Runtime routing controller", pad=24)

labels = [
    "request\nmetadata", "confidence\nprofile", "system\nstate",
    "policy\nlookup", "route\nassignment", "replica\nexecution"
]
xs = np.linspace(0.10, 0.90, len(labels))
w, h = 0.14, 0.16

for label, x in zip(labels, xs):
    rounded_box(ax, (x-w/2, 0.52), w, h, label, fontsize=12)
for x1, x2 in zip(xs[:-1], xs[1:]):
    arrow(ax, (x1+w/2, 0.60), (x2-w/2, 0.60))

rounded_box(ax, (0.27, 0.20), 0.46, 0.10,
            "Routing conditions on request structure and live infrastructure state.",
            fontsize=12)

savefig("49_runtime_routing_controller.png")

## 4. Replica routing surface

A route should change with confidence and active load. High-confidence, low-load requests can use a fast path; high-load or low-confidence requests move toward mixed or safe verification.

In [ ]:
conf = np.linspace(0.2, 0.95, 80)
load = np.linspace(0.1, 1.0, 80)
C, L = np.meshgrid(conf, load)

score_fast = C - 0.55*L
score_safe = L - C + 0.30
score_mixed = 1.0 - np.abs(C - 0.58) - 0.45*np.abs(L - 0.55)
route = np.argmax(np.stack([score_safe, score_mixed, score_fast]), axis=0)

fig, ax = plt.subplots(figsize=(9, 7))
cmap = ListedColormap(["#355f7d", "#35b779", "#fde725"])
im = ax.imshow(route, origin="lower", extent=[conf.min(), conf.max(), load.min(), load.max()],
               aspect="auto", cmap=cmap, interpolation="nearest")
ax.set_title("Replica routing depends on confidence and active load")
ax.set_xlabel("request confidence")
ax.set_ylabel("active load")
ax.text(0.33, 0.80, "safe\nverify", ha="center", va="center", fontsize=13)
ax.text(0.55, 0.56, "mixed\nroute", ha="center", va="center", fontsize=13)
ax.text(0.82, 0.36, "fast\ndraft", ha="center", va="center", fontsize=13)
ax.text(0.88, 0.20, "fast\npath", ha="center", va="center", fontsize=12)

cbar = plt.colorbar(im, ax=ax, ticks=[0, 1, 2])
cbar.ax.set_yticklabels(["safe verify", "mixed route", "fast path"])
cbar.set_label("selected route")

savefig("49_replica_routing_surface.png")

## 5. GPU pool allocation by controller policy

A deployment controller partitions GPU resources differently across regimes.

In [ ]:
regimes = ["balanced", "compute-limited", "memory-limited", "latency-protected"]
parts = ["draft", "verify", "reserve", "telemetry"]
alloc = {
    "balanced": [0.38, 0.34, 0.18, 0.10],
    "compute-limited": [0.50, 0.25, 0.15, 0.10],
    "memory-limited": [0.30, 0.36, 0.24, 0.10],
    "latency-protected": [0.28, 0.22, 0.40, 0.10],
}

fig, ax = plt.subplots(figsize=(12, 5.8))
left = np.zeros(len(regimes))
for part_i, part in enumerate(parts):
    vals = [alloc[r][part_i] for r in regimes]
    bars = ax.barh(regimes, vals, left=left, label=part)
    for j, val in enumerate(vals):
        ax.text(left[j] + val/2, j, f"{part}\n{val:.2f}", ha="center", va="center", fontsize=10)
    left += vals

ax.set_title("GPU pool allocation changes by controller policy")
ax.set_xlabel("fraction of GPU pool")
ax.set_xlim(0, 1)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.32), ncol=4)
ax.grid(axis="x", alpha=0.3)

savefig("49_gpu_pool_allocation.png")

## 6. Autoscaling gains by operating regime

Replica allocation helps most in balanced and compute-limited regimes. Latency-protected regimes saturate earlier because preserving headroom becomes the active constraint.

In [ ]:
replicas = np.arange(1, 13)
balanced = 1.0 - np.exp(-0.42*replicas)
compute = 0.95*(1 - np.exp(-0.20*replicas))
memory = 0.72*(1 - np.exp(-0.55*replicas))
latency = 0.60*(1 - np.exp(-0.65*replicas)) - 0.035*np.maximum(replicas-7, 0)
latency = np.clip(latency, 0, None)

fig, ax = plt.subplots(figsize=(12, 5.5))
for y, label in [
    (balanced, "balanced"),
    (compute, "compute-limited"),
    (memory, "memory-limited"),
    (latency, "latency-protected"),
]:
    ax.plot(replicas, y, marker="o", label=label)

ax.set_title("Autoscaling gains depend on operating regime")
ax.set_xlabel("allocated replicas")
ax.set_ylabel("throughput gain proxy")
ax.set_xticks(replicas[::1])
ax.grid(alpha=0.3)
ax.legend(loc="upper left")

savefig("49_autoscaling_regime_gains.png")

## 7. Infrastructure phase map

Infrastructure regimes are specified by reserve capacity and active load.

In [ ]:
reserve = np.linspace(0.05, 0.90, 90)
load = np.linspace(0.05, 1.00, 90)
R, L = np.meshgrid(reserve, load)

# 0 steady, 1 scale-out, 2 shed/shorten, 3 reserve
phase = np.zeros_like(R, dtype=int)
phase[(R > 0.55) & (L < 0.45)] = 3
phase[(R > 0.35) & (L >= 0.45)] = 1
phase[(R <= 0.35) & (L >= 0.65)] = 2

fig, ax = plt.subplots(figsize=(8, 7))
cmap = ListedColormap(["#440154", "#31688e", "#35b779", "#fde725"])
im = ax.imshow(phase, origin="lower", extent=[reserve.min(), reserve.max(), load.min(), load.max()],
               aspect="auto", cmap=cmap, interpolation="nearest")
ax.set_title("Infrastructure phase map")
ax.set_xlabel("reserve capacity")
ax.set_ylabel("active load")
ax.text(0.24, 0.25, "steady", ha="center", va="center", fontsize=13)
ax.text(0.61, 0.73, "scale-out", ha="center", va="center", fontsize=13)
ax.text(0.18, 0.84, "shed /\nshorten", ha="center", va="center", fontsize=13)
ax.text(0.70, 0.25, "reserve", ha="center", va="center", fontsize=13)

cbar = plt.colorbar(im, ax=ax, ticks=[0, 1, 2, 3])
cbar.ax.set_yticklabels(["steady", "scale-out", "shed/shorten", "reserve"])
cbar.set_label("infrastructure phase")

savefig("49_infrastructure_phase_map.png")

## 8. Saturation fallback path

Fallback should be observable and recoverable, not silent degradation.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5.5))
hide(ax)
ax.set_title("Saturation fallback path", pad=24)

labels = ["normal\nrouting", "load\nspike", "constraint\nactivation", "shorten\nschedule", "served\noutput"]
xs = np.linspace(0.12, 0.88, len(labels))
w, h = 0.15, 0.13

for label, x in zip(labels, xs):
    rounded_box(ax, (x-w/2, 0.62), w, h, label, fontsize=11)
for x1, x2 in zip(xs[:-1], xs[1:]):
    arrow(ax, (x1+w/2, 0.685), (x2-w/2, 0.685))

rounded_box(ax, (0.42, 0.28), 0.20, 0.10, "fallback:\nqueue / defer", fontsize=10)
rounded_box(ax, (0.64, 0.28), 0.22, 0.10, "telemetry:\nmark degraded", fontsize=10)

arrow(ax, (xs[2], 0.62), (0.52, 0.38))
arrow(ax, (0.62, 0.33), (0.64, 0.33))
arrow(ax, (0.75, 0.38), (xs[3], 0.62), rad=-0.22)

ax.text(0.50, 0.12, "Fallback makes saturation observable and recoverable.",
        ha="center", va="center", fontsize=13)

savefig("49_saturation_fallback_path.png")

## 9. Telemetry feedback loop

Telemetry closes the infrastructure loop by feeding observed latency, memory pressure, queue depth, and acceptance rate back into policy.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6.2))
hide(ax)
ax.set_title("Telemetry feedback closes the infrastructure loop", pad=24)

# two-row cycle
rounded_box(ax, (0.08, 0.60), 0.18, 0.12, "Request\nstream", fontsize=11)
rounded_box(ax, (0.36, 0.60), 0.18, 0.12, "State\nestimator", fontsize=11)
rounded_box(ax, (0.64, 0.60), 0.18, 0.12, "Routing /\nscheduling", fontsize=11)
rounded_box(ax, (0.64, 0.26), 0.18, 0.12, "Serving\nsystem", fontsize=11)
rounded_box(ax, (0.36, 0.26), 0.18, 0.12, "Telemetry\ncollector", fontsize=11)
rounded_box(ax, (0.08, 0.26), 0.18, 0.12, "Policy\nupdate", fontsize=11)

arrow(ax, (0.26, 0.66), (0.36, 0.66))
arrow(ax, (0.54, 0.66), (0.64, 0.66))
arrow(ax, (0.73, 0.60), (0.73, 0.38))
arrow(ax, (0.64, 0.32), (0.54, 0.32))
arrow(ax, (0.36, 0.32), (0.26, 0.32))
arrow(ax, (0.17, 0.38), (0.17, 0.60))
arrow(ax, (0.44, 0.38), (0.45, 0.60), rad=-0.18)

ax.text(0.50, 0.10,
        "Observed latency, memory pressure, queue depth, and acceptance rate update the next policy.",
        ha="center", va="center", fontsize=12)

savefig("49_telemetry_feedback_loop.png")

## 10. Final synthesis

Notebook 49 bridges controller policy and deployment-scale serving infrastructure.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5.5))
hide(ax)
ax.set_title("Infrastructure specifies adaptive serving", pad=24)

labels = [
    "operating\nregime", "controller\npolicy", "routing\nstate",
    "replica\nallocation", "telemetry\nfeedback", "adaptive\nserving"
]
xs = np.linspace(0.10, 0.90, len(labels))
w, h = 0.14, 0.15

for label, x in zip(labels, xs):
    rounded_box(ax, (x-w/2, 0.55), w, h, label, fontsize=12)
for x1, x2 in zip(xs[:-1], xs[1:]):
    arrow(ax, (x1+w/2, 0.625), (x2-w/2, 0.625))

rounded_box(ax, (0.29, 0.22), 0.42, 0.09,
            "Notebook 49 bridges scheduling policy and deployment-scale serving infrastructure.",
            fontsize=12)

savefig("49_final_synthesis.png")

## Export results

This cell writes a ZIP containing all Notebook 49 figures and a small manifest.

In [ ]:
manifest = {
    "notebook": "49_adaptive_serving_infrastructure_v3",
    "title": "Adaptive Serving Infrastructure",
    "statement": "Adaptive serving infrastructure turns operating-regime decisions into deployable routing, scaling, fallback, and telemetry loops.",
    "figures": sorted(p.name for p in FIG_DIR.glob("49_*.png")),
}

manifest_path = ROOT / "results" / "notebook49_manifest.json"
manifest_path.parent.mkdir(parents=True, exist_ok=True)
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(manifest_path, arcname="notebook49_manifest.json")
    for fig_path in sorted(FIG_DIR.glob("49_*.png")):
        zf.write(fig_path, arcname=f"figures/{fig_path.name}")

print(f"Wrote {ZIP_PATH}")
display(FileLink(str(ZIP_PATH)))